# Citadel Model Aliases — three-surface feature validation

Configure priority and weighted aliases using deployed chat models with inferred or overridden operations.
Test direct controls, alias/resolved-model headers, controlled priority, weighted membership,
discovery filtering, and RBAC across Universal, Azure OpenAI and Unified AI APIs.

**Shared routing side effect:** onboarding regenerates shared policy fragments. Run All uses the
complete azd backend configuration and optional aliases/global settings, verifies live backend coverage,
and deploys by default. A complete source parameter JSON override is optional, not a prerequisite.
This notebook never deletes/reverts shared backend or alias configuration. Extra statistical
sampling is opt-in; no exact split is demanded from a small random sample.

## 1. Editable source configuration and bounded settings

### 1. Editable source configuration and bounded settings

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Scenario-specific input examples


`source_parameters_file` is the complete backend-routing contract, including
existing aliases and circuit-breaker/affinity settings. Example:
`source_parameters_file = root / "validation" / ".runs" / "backend-onboarding" / "your-run-id" / "backends" / "main.bicepparam"`.
Set `source_configuration_complete = True` and `approve_shared_routing = True`
only after reviewing that source. `alias_member_overrides = ["my-chat-a", "my-chat-b"]`
pins compatible members; `None` selects from resolved operations.
`priority_control_confirmed = True` asserts the selected primary is healthy for
the deterministic control test. Weighted sampling is optional:
`enable_distribution_sampling = True; distribution_requests = 50`.

**Run All defaults:** azd supplies the complete backend inventory and optional aliases/global settings; APIM supplies its user-assigned identity. Provisioning is enabled, without confirmation prompts. An explicit source file remains optional and authoritative. Skip keywords affect requests, never deployment inventory. Missing optional alias-model prerequisites produce SKIP and prevent dependent deployments/tests.


In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(root / "shared"))
import utils
with utils.notebook_step('1. Editable source configuration and bounded settings', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    from pathlib import Path
    import sys, json, time, uuid
    from urllib.parse import quote
    from requests.structures import CaseInsensitiveDict
    from validationtools import ValidationContext, ValidationRun, ModelCatalog, RequestTransport, ValidationBlocked
    from apimtools import APIMClientTool
    from contracttools import scoped_deployment_name
    from notebookdefaults import backend_source, source_catalog
    from contracttools import access_policy, prepare_artifact, deploy_contract
    from consumerhelpers import (
        access_artifact, contract_identifiers, cli_json, chat_content, error_code,
        complete_backend_parameters, load_contract_manifest, delete_owned_product, product_exists,
        review_backend_artifact,
    )
    environment = None
    from_azd = True
    overrides = {}  # Explicit canonical config/capabilities override the selected azd environment.
    run_id = time.strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
    inference_api_version = "2024-05-01-preview"
    openai_api_version = "2024-12-01-preview"
    cleanup_enabled = False

    source_parameters_file = None  # Optional complete source override; otherwise use the selected azd inventory.
    source_configuration_complete = True
    approve_shared_routing = True
    direct_model_override = None
    alias_member_overrides = None  # Optional distinct deployed, chat-capable names.
    priority_control_confirmed = False  # Confirm first member healthy and fallback not being fault-injected.
    enable_distribution_sampling = False
    distribution_requests = 30  # bounded to 1..100
    run = ValidationRun("model-aliases")
    run_dir = root / "validation" / ".runs" / "model-aliases" / run_id

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


## 2. Resolve members from the catalog and inspect live routing inventory
Operations are inferred automatically unless model metadata or operation_overrides explicitly replaces them. Missing optional alias-model prerequisites produce SKIP.

### 2. Resolve members from the catalog and inspect live routing inventory

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('2. Resolve members from the catalog and inspect live routing inventory', inputs={'source_parameters_file': globals().get('source_parameters_file'), 'overrides': globals().get('overrides'), 'environment': globals().get('environment'), 'from_azd': globals().get('from_azd'), 'run_dir': globals().get('run_dir'), 'source_configuration_complete': globals().get('source_configuration_complete')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=from_azd,
        operation_overrides=operation_overrides, skipped_models=skipped_models)
    utils.require_inputs({f"gateway.{key}": context.config.get(key)
        for key in ("subscription_id", "resource_group", "location", "apim_name")})
    transport = RequestTransport(timeout=60)
    apim = APIMClientTool(context.config["resource_group"], context.config["apim_name"])
    apim.initialize_context(context.account, enumerate_subscriptions=False)
    source_parameters = backend_source(context, apim, source_parameters_file,
        overrides.get("backend_template_options"))
    context.catalog = source_catalog(context, source_parameters)
    before_backends, before_pools = apim.get_backends()
    compatible_types = {"ai-foundry", "azure-openai", "aws-bedrock-mantle", "gemini-openai"}
    candidates = [name for name in context.catalog.models_for("chat")
        if any(entry.name == name and entry.backend_type in compatible_types for entry in context.catalog.entries)]
    model_aliases = []
    members = []
    aliases_ready = False
    try:
        if len(candidates) < 2:
            raise ValueError("Two distinct eligible OpenAI-compatible chat models are needed for alias coverage.")
        direct_test_model = context.catalog.select("chat", override=direct_model_override or candidates[0])
        if direct_test_model not in candidates:
            raise ValueError("Direct control needs an OpenAI-compatible deployed chat model.")
        members = list(alias_member_overrides or candidates[:2])
        if any(member not in candidates for member in members):
            raise ValueError("Alias members must be eligible OpenAI-compatible deployed chat models.")
        members = context.catalog.alias_members(members, "chat", min_members=2)
        model_aliases = [
            {"name": "validation-priority", "models": members, "strategy": "priority"},
            {"name": "validation-weighted", "models": members, "strategy": "weighted",
             "weights": [70] + [30] * (len(members) - 1)},
        ]
        aliases_ready = True
    except (ValueError, ValidationBlocked) as exc:
        if direct_model_override or alias_member_overrides:
            raise
        run.skip("alias-member-selection", str(exc))
    utils.format_output("Alias initialization", {"eligible_models": candidates,
        "model_aliases": model_aliases, "deployment_backends": len(source_parameters["llmBackendConfig"]),
        "scenario_ready": aliases_ready}, status="info", part="RESULT")
    run_dir.mkdir(parents=True, exist_ok=True)


## 3. Generate and preview the editable .bicepparam
Preserve all existing aliases and every backend field; reject inline secrets and hidden backend replacements.

### 3. Generate and preview the editable .bicepparam

Generate the editable parameter artifact for 3. Generate and preview the editable .bicepparam and display the proposed configuration without deploying it.

**Input:** `source_parameters`, `model_aliases`, `source_configuration_complete`, `item`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('3. Generate and preview the editable .bicepparam', inputs={'source_parameters': globals().get('source_parameters'), 'model_aliases': globals().get('model_aliases'), 'source_configuration_complete': globals().get('source_configuration_complete'), 'item': globals().get('item'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        try:
            backend_parameters = complete_backend_parameters(
                source_parameters, [], confirmed_complete=source_configuration_complete,
            )
            aliases_by_name = {item["name"]: item for item in backend_parameters["modelAliases"]}
            aliases_by_name.update((item["name"], item) for item in model_aliases)
            backend_parameters["modelAliases"] = list(aliases_by_name.values())
            expected_backend_ids = {item["backendId"] for item in backend_parameters["llmBackendConfig"]}
            missing_source_ids = {item["name"] for item in before_backends} - expected_backend_ids
            if missing_source_ids:
                raise ValueError(f"Source omitted live backends: {sorted(missing_source_ids)}")
            backend_parameters["apim"] = {
                "subscriptionId": context.config["subscription_id"],
                "resourceGroupName": context.config["resource_group"], "name": apim.apim_resource_name,
            }
            if not backend_parameters.get("apimManagedIdentity"):
                raise ValueError("Full source must include apimManagedIdentity coordinates.")
            backend_artifact = prepare_artifact(
                root / "bicep" / "infra" / "llm-backend-onboarding" / "main.bicep",
                backend_parameters, run_dir / "backends",
            )
            utils.format_output('3. Generate and preview the editable .bicepparam', {'value_1': backend_artifact.preview()}, status='info', part='RESULT')
            utils.format_output('3. Generate and preview the editable .bicepparam', {'value_1': "Editable primary deployment input:", 'value_2': backend_artifact.params_file}, status='info', part='RESULT')
        except Exception as exc:
            run.blocked("shared-routing-preflight", str(exc))
            raise


## 4. Deploy after reviewing side effects and any parameter edits
Shared backends and previous aliases persist. Compiler validates the actual edited parameter file.

### 4. Deploy after reviewing side effects and any parameter edits

Deploy the reviewed artifact for 4. Deploy after reviewing side effects and any parameter edits, then inspect its real resource outputs and ownership.

**Input:** `approve_shared_routing`, `backend_artifact`, `source_parameters`, `backend_parameters`, `item`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('4. Deploy after reviewing side effects and any parameter edits', inputs={'approve_shared_routing': globals().get('approve_shared_routing'), 'backend_artifact': globals().get('backend_artifact'), 'source_parameters': globals().get('source_parameters'), 'backend_parameters': globals().get('backend_parameters'), 'item': globals().get('item')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        if not approve_shared_routing:
            run.blocked("shared-routing-deploy", "Review the full .bicepparam and set approve_shared_routing=True.")
            raise RuntimeError("Shared routing deployment has not been approved.")
        effective = review_backend_artifact(backend_artifact, source_parameters, backend_parameters["apim"],
                                            {item["name"] for item in before_backends},
                                            editable_aliases={item["name"] for item in model_aliases})
        expected_backend_ids = {item["backendId"] for item in effective["llmBackendConfig"]}
        backend_outputs = deploy_contract(backend_artifact.params_file, context.config["location"],
                                         scoped_deployment_name("validation-alias-backends", apim.apim_service_id))
        after_backends, after_pools = apim.get_backends()
        live_ids = {item["name"] for item in after_backends}
        run.check("backend-inventory", expected_backend_ids <= live_ids,
                  expected=sorted(expected_backend_ids), actual=sorted(live_ids))
        fragment = apim.client.policy_fragment.get(
            context.config["resource_group"], apim.apim_resource_name, "resolve-model-alias",
        )
        run.check("alias-fragment", all(item["name"] in fragment.value for item in model_aliases),
                  expected="Both new aliases in deployed fragment", actual="live policy inspection")


## 5. Generate a least-privilege access contract
Grant only the alias names and direct control. Underlying members not directly granted remain hidden by discovery RBAC.

### 5. Generate a least-privilege access contract

Prepare the 5. Generate a least-privilege access contract state shown below for its subsequent feature-specific requests and assertions.

**Input:** `direct_test_model`, `run_dir`, `run_id`, `item`

**Output:** `use_case`, `allowed_models`, `product_id`, `policy_xml`, `contract_artifact` are available to the following cells.


In [ ]:
with utils.notebook_step('5. Generate a least-privilege access contract', inputs={'direct_test_model': globals().get('direct_test_model'), 'run_dir': globals().get('run_dir'), 'run_id': globals().get('run_id'), 'item': globals().get('item')}, expected='`use_case`, `allowed_models`, `product_id`, `policy_xml`, `contract_artifact` are available to the following cells.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        use_case = {"businessUnit": "Validation", "useCaseName": "Alias", "environment": "DEV"}
        allowed_models = [item["name"] for item in model_aliases] + [direct_test_model]
        product_id = f"LLM-{use_case['businessUnit']}-{use_case['useCaseName']}-{use_case['environment']}"
        policy_xml = access_policy(allowed_models, 10000, 100000, "Daily",
                                  inbound_extra='<set-variable name="enableResponseHeaders" value="@(true)" />')
        utils.format_output('5. Generate a least-privilege access contract', {'policy_xml': policy_xml}, status='info', part='RESULT')
        contract_artifact = access_artifact(root, run_dir / "access", context, apim, use_case,
                                           ["universal-llm-api", "azure-openai-api", "unified-ai-api"], policy_xml)
        utils.format_output('5. Generate a least-privilege access contract', {'value_1': contract_artifact.preview()}, status='info', part='RESULT')


## 6. Deploy contract, retrieve its exact key, discover independent API descriptors

### 6. Deploy contract, retrieve its exact key, discover independent API descriptors

Deploy the reviewed artifact for 6. Deploy contract, retrieve its exact key, discover independent API descriptors, then inspect its real resource outputs and ownership.

**Input:** `product_id`, `contract_artifact`, `run_id`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('6. Deploy contract, retrieve its exact key, discover independent API descriptors', inputs={'product_id': globals().get('product_id'), 'contract_artifact': globals().get('contract_artifact'), 'run_id': globals().get('run_id')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        product_preexists = product_exists(context, apim, product_id)
        contract_outputs = deploy_contract(contract_artifact.params_file, context.config["location"], scoped_deployment_name("validation-alias-access", apim.apim_service_id))
        product_id, subscription_name = contract_identifiers(contract_outputs, product_id)
        api_key = apim.get_subscription_key(subscription_name)
        contract_artifact.record_resource(f"{apim.apim_service_id}/products/{product_id}", created=not product_preexists)
        endpoints = {label: apim.discover_api_exact(api_path)["url"]
                     for label, api_path in [("universal", "models"), ("openai", "openai"), ("unified", "unified-ai")]}


## 7. Explicit routes, payloads and response proof
Body text, alias identity and resolved membership must all match; missing debug headers fail rather than silently passing.

### 7. Explicit routes, payloads and response proof

Exercise 7. Explicit routes, payloads and response proof and compare the actual response/evidence with the expected feature behavior.

**Input:** `openai_api_version`, `inference_api_version`, `endpoints`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('7. Explicit routes, payloads and response proof', inputs={'openai_api_version': globals().get('openai_api_version'), 'inference_api_version': globals().get('inference_api_version'), 'endpoints': globals().get('endpoints')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        def record(case, result, expected_status=200, body_check=None, **evidence):
            result.headers = CaseInsensitiveDict(result.headers)
            valid_body = True
            if body_check is not None:
                try:
                    valid_body = bool(body_check(result.json_body))
                except (TypeError, ValueError, KeyError, IndexError, AttributeError):
                    valid_body = False
            run.check(
                case, not result.error and result.status_code == expected_status and valid_body,
                expected=f"HTTP {expected_status}; expected response shape/policy evidence",
                actual=result.error or str(result.status_code),
                duration=result.duration,
                headers={k: v for k, v in result.headers.items() if k.lower().startswith("uaig-")},
                **evidence,
            )
            return result

        messages = [{"role": "user", "content": "Reply with the single word OK."}]
        def call_chat(surface, model):
            if surface == "openai":
                url = f"{endpoints[surface]}/deployments/{quote(model, safe='')}/chat/completions?api-version={openai_api_version}"
                body = {"messages": messages}
            else:
                operation = "models/chat/completions" if surface == "unified" else "chat/completions"
                url = f"{endpoints[surface]}/{operation}?api-version={inference_api_version}"
                body = {"model": model, "messages": messages}
            return transport.request("POST", url, headers={"api-key": api_key}, json=body)

        def validate_chat(surface, model, alias=None):
            result = call_chat(surface, model)
            case = f"{surface}/{model}"
            record(case, result, body_check=lambda data: bool(chat_content(data)))
            if alias:
                run.check(f"{case}/resolution", result.headers.get("UAIG-Alias") == model
                          and result.headers.get("UAIG-Resolved-Model") in alias["models"],
                          expected={"alias": model, "members": alias["models"]},
                          actual={k: result.headers.get(k) for k in ("UAIG-Alias", "UAIG-Resolved-Model")})
            else:
                run.check(f"{case}/direct-control", not result.headers.get("UAIG-Alias")
                          and result.headers.get("UAIG-Model-Id") == model,
                          expected="Direct model; alias resolver no-op", actual=result.headers.get("UAIG-Model-Id"))
            return result


## 8. Discovery filtering and first-class alias entries — all three APIs

### 8. Discovery filtering and first-class alias entries — all three APIs

Exercise 8. Discovery filtering and first-class alias entries — all three APIs and compare the actual response/evidence with the expected feature behavior.

**Input:** `model_aliases`, `endpoints`, `result`, `item`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('8. Discovery filtering and first-class alias entries — all three APIs', inputs={'model_aliases': globals().get('model_aliases'), 'endpoints': globals().get('endpoints'), 'result': globals().get('result'), 'item': globals().get('item')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        for surface, endpoint in endpoints.items():
            result = transport.request("GET", f"{endpoint}/deployments", headers={"api-key": api_key})
            record(f"{surface}/discovery", result,
                   body_check=lambda data: {item["name"] for item in data["value"]} == set(allowed_models))
            entries = (result.json_body or {}).get("value", [])
            for alias in model_aliases:
                entry = next((item for item in entries if item.get("name") == alias["name"]), {})
                run.check(f"{surface}/discovery/{alias['name']}",
                          entry.get("type") == "alias" and entry.get("properties", {}).get("model", {}).get("format") == "Alias"
                          and bool(entry.get("properties", {}).get("capabilities", {}).get("description")),
                          expected="Alias type, format and description", actual=entry)


## 9. Direct controls, controlled priority and weighted membership
Priority's first-member expectation requires an explicit healthy-primary control. Weighted calls assert membership, not an exact random split.

### 9. Direct controls, controlled priority and weighted membership

Exercise 9. Direct controls, controlled priority and weighted membership and compare the actual response/evidence with the expected feature behavior.

**Input:** `endpoints`, `model_aliases`, `priority_control_confirmed`, `surface`, `direct_test_model`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('9. Direct controls, controlled priority and weighted membership', inputs={'endpoints': globals().get('endpoints'), 'model_aliases': globals().get('model_aliases'), 'priority_control_confirmed': globals().get('priority_control_confirmed'), 'surface': globals().get('surface'), 'direct_test_model': globals().get('direct_test_model')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        for surface in endpoints:
            validate_chat(surface, direct_test_model)
        priority_alias, weighted_alias = model_aliases
        for surface in endpoints:
            result = validate_chat(surface, priority_alias["name"], priority_alias)
            if priority_control_confirmed:
                run.check(f"{surface}/priority-first", result.headers.get("UAIG-Resolved-Model") == members[0],
                          expected=members[0], actual=result.headers.get("UAIG-Resolved-Model"))
            else:
                run.skip(f"{surface}/priority-first", "Healthy-primary ordering is not assumed; resolved membership is checked above.")
            validate_chat(surface, weighted_alias["name"], weighted_alias)


## 10. Optional bounded distribution sampling
All sampled responses must still have valid alias and resolved-member evidence. Observed proportions are descriptive only.

### 10. Optional bounded distribution sampling

Prepare the 10. Optional bounded distribution sampling state shown below for its subsequent feature-specific requests and assertions.

**Input:** `enable_distribution_sampling`, `distribution_requests`, `weighted_alias`, `result`

**Output:** `counts`, `index`, `result` are available to the following cells.


In [ ]:
with utils.notebook_step('10. Optional bounded distribution sampling', inputs={'enable_distribution_sampling': globals().get('enable_distribution_sampling'), 'distribution_requests': globals().get('distribution_requests'), 'weighted_alias': globals().get('weighted_alias'), 'result': globals().get('result')}, expected='`counts`, `index`, `result` are available to the following cells.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        from collections import Counter
        if enable_distribution_sampling:
            if not 1 <= distribution_requests <= 100:
                raise ValueError("distribution_requests must be within 1..100.")
            counts = Counter()
            for index in range(distribution_requests):
                result = validate_chat("universal", weighted_alias["name"], weighted_alias)
                counts[result.headers.get("UAIG-Resolved-Model", "(missing)")] += 1
            utils.format_output('10. Optional bounded distribution sampling', {'value_1': "Observed weighted distribution:", 'value_2': dict(counts), 'value_3': "configured:", 'value_4': weighted_alias["weights"]}, status='info', part='RESULT')
        else:
            run.skip("weighted-distribution", "Additional statistical sampling is disabled.")


## 11. Underlying member denied — all three APIs
Require the exact unauthorized_model_access policy error; a generic provider error is not RBAC proof.

### 11. Underlying member denied — all three APIs

Collect the final 11. Underlying member denied — all three APIs verdict without discarding earlier failed or blocked required cases.

**Input:** `endpoints`, `members`, `allowed_models`, `surface`

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('11. Underlying member denied — all three APIs', inputs={'endpoints': globals().get('endpoints'), 'members': globals().get('members'), 'allowed_models': globals().get('allowed_models'), 'surface': globals().get('surface')}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        denied_model = next((model for model in members if model not in allowed_models), None)
        if not denied_model:
            run.skip("model-rbac", "No eligible alias member remains outside direct contract grants.")
        else:
            for surface in endpoints:
                record(f"{surface}/model-rbac", call_chat(surface, denied_model), 403,
                       body_check=lambda data: error_code(data) == "unauthorized_model_access")
        run.finish()


## 12. Opt-in cleanup
Only an access product/subscription newly created by this run is eligible. Reused products and shared backend and alias routing are retained.

### 12. Opt-in cleanup

Apply the explicitly enabled 12. Opt-in cleanup action only to resources owned by this run.

**Input:** `cleanup_enabled`, `product_id`, `contract_artifact`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('12. Opt-in cleanup', inputs={'cleanup_enabled': globals().get('cleanup_enabled'), 'product_id': globals().get('product_id'), 'contract_artifact': globals().get('contract_artifact')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    if not aliases_ready:
        utils.format_output("Optional scenario", {"reason": "Eligible model prerequisites are absent; this cell performs no deployment or test."}, status="SKIP", part="RESULT")
    if aliases_ready:
        utils.format_output('12. Opt-in cleanup', {'value_1': "Owned cleanup candidates:", 'value_2': contract_artifact.cleanup_candidates()}, status='info', part='RESULT')
        if cleanup_enabled:
            delete_owned_product(context, apim, product_id, artifact=contract_artifact)
        else:
            utils.format_output('12. Opt-in cleanup', {'value_1': "Cleanup disabled; shared routing is never reverted."}, status='info', part='RESULT')
    if not aliases_ready:
        utils.format_output("Alias scenario", {"reason": "No eligible alias-model prerequisites; no tests executed."}, status="SKIP", part="RESULT")
